In [ ]:
# Vulkan for SAPIEN (from ManiSkill's quickstart notebook)
!mkdir -p /usr/share/vulkan/icd.d
!wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/nvidia_icd.json
!wget -q https://raw.githubusercontent.com/haosulab/ManiSkill/main/docker/10_nvidia.json
!mv nvidia_icd.json /usr/share/vulkan/icd.d
!mv 10_nvidia.json /usr/share/glvnd/egl_vendor.d/10_nvidia.json
!apt-get install -y --no-install-recommends libvulkan-dev > /dev/null

In [ ]:
!python --version
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/so101-runs'
!rm -rf /content/so101 && git clone -q -b main https://github.com/HipsterBrown/so101-policy-learning.git /content/so101
!git -C /content/so101 log -1 --oneline  # record this commit with each run
%cd /content/so101
# mplib (ManiSkill's motion-planning examples only; unused here) has no Python 3.13 wheel, so drop it via a uv override.
!pip install -q uv
!echo "mplib ; sys_platform == 'never'" > /tmp/overrides.txt
!uv pip install --system -q -e . --override /tmp/overrides.txt
import site; site.main()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {OUT}

In [ ]:
# Phase 2 gate A: Reach still >= 0.95 with the new obs-norm + ent_coef defaults.
# --no-partial-reset: ManiSkill's BaseEnv.reset resets controllers with _reset_mask already set back to all envs,
# so a partial reset clobbers every env's target-delta joint target mid-episode. All envs reset together instead.
# ppo.py prints `control_mode=... obs_dim=...` before training: expect pd_joint_target_delta_pos and (32,)
!python scripts/ppo.py --env_id SO101Reach-v1 --seed 1 --exp-name reach-p2val-s1 \
  --num_envs 2048 --update_epochs 8 --num_minibatches 32 --num_steps 20 --total_timesteps 10_000_000 \
  --eval_freq 3 --num_eval_envs 100 --num_eval_steps 50 \
  --output-dir {OUT} --variant p2-validation --results-csv {OUT}/results.csv --no-capture-video --no-partial-reset

In [ ]:
# Phase 2 throughput: 1M Lift steps; read charts/SPS in TensorBoard and record SPS + minutes per 10M in the spec.
# If a 30M-step seed would exceed 90 min: set solver_position_iterations=40 in so101/envs.py, push, re-run cell 2 (fresh clone).
!python scripts/ppo.py --env_id SO101Lift-v1 --seed 1 --exp-name lift-sps \
  --num_envs 1024 --update_epochs 8 --num_minibatches 32 --total_timesteps 1_000_000 \
  --eval_freq 5 --num_eval_envs 100 --num_eval_steps 100 \
  --output-dir {OUT} --variant sps --no-capture-video --no-partial-reset

In [ ]:
# Phase 2 gate B: 3 seeds, <= 30M steps each. Fallback trigger (a): stop a seed whose eval/success_once < 0.20 at 20M.
# A disconnect restarts that seed from scratch: delete {OUT}/lift-baseline-s{SEED} first.
# An unfinished seed's folder must be deleted before re-running, or its stale files mix with the new run.
TOTAL = 30_000_000
import os
for SEED in [1, 2, 3]:
    if os.path.exists(f"{OUT}/lift-baseline-s{SEED}/final_ckpt.pt"):
        continue  # finished seed: re-running the cell after a disconnect/interrupt resumes with the next one
    !python scripts/ppo.py --env_id SO101Lift-v1 --seed {SEED} --exp-name lift-baseline-s{SEED} \
      --num_envs 1024 --update_epochs 8 --num_minibatches 32 --total_timesteps {TOTAL} \
      --eval_freq 5 --num_eval_envs 100 --num_eval_steps 100 \
      --output-dir {OUT} --variant baseline --results-csv {OUT}/results.csv --no-capture-video --no-partial-reset